# Week 5: AI-Assisted Triage Data Exploration

This notebook explores the Week 5 ED triage dataset and generates the profiling tables, cleaned local dataset, and visual dashboard used in the feasibility memo. The goal is to answer whether the dataset is good enough to begin baseline modelling in Week 6.

Privacy note: the raw and cleaned patient-level tables are stored locally under `data/raw/` and `data/processed/`, which are ignored by Git.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_PATH = ROOT / 'data' / 'raw' / 'full_data.parquet'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'week5_triage_cleaned.parquet'
OUTPUT_DIR = ROOT / 'week-5' / 'outputs'
SUMMARY_DIR = ROOT / 'week-5' / 'assignments'

print(RAW_PATH)

In [ ]:
df = pd.read_parquet(RAW_PATH)
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.head()

## Data Categories

The dataset includes encounter context, demographics, triage acuity, prior utilisation, diagnosis/history indicators, labs, urinalysis, triage vitals, subsequent vitals, imaging counts, medication groups, surgery history, and one-hot chief complaint indicators. These categories matter because triage level is not driven by one value alone; it reflects physiological stability, presenting complaint, age/risk context, and available clinical history.

In [ ]:
dtype_summary = df.dtypes.astype(str).value_counts().rename_axis('dtype').reset_index(name='column_count')
dtype_summary

In [ ]:
missing = (
    df.isna().mean().mul(100).rename('missing_pct').to_frame()
    .assign(missing_count=df.isna().sum().values, dtype=df.dtypes.astype(str).values)
    .sort_values('missing_pct', ascending=False)
)
missing.head(25)

In [ ]:
esi = pd.to_numeric(df['esi'].astype(str), errors='coerce')
esi.value_counts(dropna=False).sort_index().to_frame('count')

In [ ]:
vital_ranges = {
    'triage_vital_hr': (25, 250),
    'triage_vital_sbp': (50, 260),
    'triage_vital_dbp': (20, 160),
    'triage_vital_rr': (5, 80),
    'triage_vital_o2': (50, 100),
    'triage_vital_temp': (85, 110),
}
outlier_rows = []
for col, (lo, hi) in vital_ranges.items():
    s = pd.to_numeric(df[col], errors='coerce')
    outside = s.notna() & ~s.between(lo, hi)
    outlier_rows.append({
        'feature': col, 'plausible_low': lo, 'plausible_high': hi,
        'observed_min': s.min(), 'observed_max': s.max(),
        'missing_pct': s.isna().mean() * 100,
        'outside_plausible_count': int(outside.sum()),
        'outside_plausible_pct': outside.mean() * 100,
    })
pd.DataFrame(outlier_rows)

In [ ]:
clean = df.copy()
clean['esi_numeric'] = esi
clean['acuity_score'] = 6 - clean['esi_numeric']
for col, (lo, hi) in vital_ranges.items():
    flag = f'{col}_outside_plausible_range'
    clean[flag] = clean[col].notna() & ~pd.to_numeric(clean[col], errors='coerce').between(lo, hi)
    clean.loc[clean[flag], col] = np.nan

arrival = clean['arrivalmode'].astype(str).str.lower()
clean['arrivalmode_ambulance_or_helicopter'] = arrival.str.contains('ambulance|helicopter|ems|critical', regex=True).astype(float)
clean.to_parquet(PROCESSED_PATH, index=False)
print(PROCESSED_PATH)

In [ ]:
cc_cols = [c for c in clean.columns if c.startswith('cc_')]
cc_prev = clean[cc_cols].mean().sort_values(ascending=False).head(20).mul(100)
cc_prev.to_frame('prevalence_pct')

In [ ]:
corr_rows = []
for col in clean.columns:
    if col in {'esi', 'esi_numeric', 'acuity_score'} or not pd.api.types.is_numeric_dtype(clean[col]):
        continue
    pair = pd.concat([pd.to_numeric(clean[col], errors='coerce'), clean['acuity_score']], axis=1).dropna()
    if len(pair) >= 1000 and pair.iloc[:, 0].nunique() >= 2:
        corr = pair.iloc[:, 0].corr(pair.iloc[:, 1])
        if pd.notna(corr):
            corr_rows.append({'feature': col, 'correlation_with_acuity': corr, 'abs_correlation': abs(corr), 'non_missing_pct': len(pair)/len(clean)*100})
correlations = pd.DataFrame(corr_rows).sort_values('abs_correlation', ascending=False)
correlations.head(20)

In [ ]:
shortlist = pd.read_csv(SUMMARY_DIR / 'week5_top10_feature_shortlist.csv')
shortlist

## Dashboard Outputs

The generated visual outputs are saved in `week-5/outputs/`: missingness, ESI distribution, race/ethnicity distribution, chief complaints, vital signs by ESI, and feature-acuity correlations. These files support the feasibility memo and give one ready-made image for Discord.